# Notebook 2 Bases de Datos Avanzadas - MongoDB

## 1. Librerías y dependencias

In [1]:
%pip install pymongo
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: C:\Users\arand\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: C:\Users\arand\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
from pymongo import MongoClient, ReadPreference
import hashlib
import subprocess
import os
import urllib.request
import numpy as np
from sentence_transformers import SentenceTransformer


Descargamos el modelo Transformer que utilizaremos para calcular los embedding de los discursos.

Descomentar para descargar el modelo. (Si ya esta descargado no descomentar)

In [3]:
if not os.path.exists("./modelo_local/model.safetensors"):
    print("Modelo no encontrado, descargando...")
    
    base_url = "https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/"
    os.makedirs("./modelo_local/1_Pooling", exist_ok=True)

    archivos = [
        "config.json",
        "tokenizer_config.json",
        "tokenizer.json",
        "sentencepiece.bpe.model",
        "modules.json",
        "sentence_bert_config.json",
        "special_tokens_map.json",
        "model.safetensors",
        "1_Pooling/config.json"
    ]

    for archivo in archivos:
        destino = f"./modelo_local/{archivo}"
        print(f"Descargando {archivo}...")
        urllib.request.urlretrieve(base_url + archivo, destino)
        size = os.path.getsize(destino)
        print(f"  ✅ {archivo} ({size/1024/1024:.1f} MB)")

    print("\n✅ Modelo descargado")
else:
    print("✅ Modelo ya existe, omitiendo descarga")

Modelo no encontrado, descargando...
Descargando config.json...
  ✅ config.json (0.0 MB)
Descargando tokenizer_config.json...
  ✅ tokenizer_config.json (0.0 MB)
Descargando tokenizer.json...
  ✅ tokenizer.json (8.7 MB)
Descargando sentencepiece.bpe.model...
  ✅ sentencepiece.bpe.model (4.8 MB)
Descargando modules.json...
  ✅ modules.json (0.0 MB)
Descargando sentence_bert_config.json...
  ✅ sentence_bert_config.json (0.0 MB)
Descargando special_tokens_map.json...
  ✅ special_tokens_map.json (0.0 MB)
Descargando model.safetensors...
  ✅ model.safetensors (448.8 MB)
Descargando 1_Pooling/config.json...
  ✅ 1_Pooling/config.json (0.0 MB)

✅ Modelo descargado


## 2. Conexión y creación de BD Mongo llamada "Política"

In [4]:
try:
    client = MongoClient(
        "mongodb://mongo-primary:30001,mongo-secondary1:30002,mongo-secondary2:30003/?replicaSet=rs-politica&readPreference=primary&ssl=false"
    )

    client.admin.command('ping')
    print(client.primary)
    print(client.secondaries)

    db = client["Política"]
    coleccion = db["Discursos"]
    
    print("Conexión exitosa! BD y colección creadas")
except Exception as e:
    print(f"Error de conexión: {e}")

('mongo-primary', 30001)
{('mongo-secondary1', 30002), ('mongo-secondary2', 30003)}
Conexión exitosa! BD y colección creadas


In [5]:
# Insertamos un dato de prueba
coleccion.insert_one({"titulo": "Prueba", "autor": "Bruno"})

InsertOneResult(ObjectId('6a1caf15146089a75092958c'), acknowledged=True)

> Conéctate a MongoDB desde la terminal
```bash
docker exec -it mongo-primary mongo --port 30001
```

> Ahora probar los comandos de MongoDB en la terminal.
```javascript
show dbs                        // ver todas las bases de datos
use Política                    // seleccionar la BD
show collections                // ver colecciones
db.Discursos.find()             // ver documentos
db.Discursos.find().pretty()    // ver documentos formateado
db.Discursos.deleteMany({})     // eliminar todos los elementos de la base de datos
db.Discursos.countDocuments({}) // contamos la cantidad de documentos dentro de la colección Discrusos
```

In [6]:
# Eliminamos el/los dato de prueba
coleccion.delete_many({"titulo": "Prueba", "autor": "Bruno"})

DeleteResult({'n': 1, 'electionId': ObjectId('7fffffff0000000000000001'), 'opTime': {'ts': Timestamp(1780264730, 1), 't': 1}, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1780264730, 1), 'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00', 'keyId': 0}}, 'operationTime': Timestamp(1780264730, 1)}, acknowledged=True)

> Volver a probar el comando para buscar discursos, verán que ya no hay

Testeamos que los 3 nodos se encuentren activos

In [113]:
primary    = MongoClient("mongodb://localhost:30001/?directConnection=true")
secondary1 = MongoClient("mongodb://localhost:30002/?directConnection=true")
secondary2 = MongoClient("mongodb://localhost:30003/?directConnection=true")

for nombre, cliente in [("Primary", primary), ("Secondary1", secondary1), ("Secondary2", secondary2)]:
    resultado = cliente.admin.command("ping")
    print(f"{'✅' if resultado['ok'] else '❌'} {nombre}")

✅ Primary
✅ Secondary1
✅ Secondary2


## 3. Inserción de documentos a Mongo


> Tanto para generar los embedding de todos los discursos como para luego generar embedding del texto al cual queremos buscarle similitudes, usaremos el modelo **multi-lenguaje** 'MiniLM-L12-v2' de la librería Sentence Transformers.

In [114]:
# Guardamos la ruta de la carpeta que contiene los .txt
ruta_carpeta = "./DiscursosOriginales"

# Cargamos el modelo de Sentence-Transformers
modelo_transformer = SentenceTransformer('./modelo_local/')

print("==============  Inicio del procesamiento de discursos  ==============")

# Recorremos cada archivo de texto dentro de la carpeta DiscursosOriginales
for archivo in os.listdir(ruta_carpeta):
    if archivo.endswith(".txt"):
        ruta_completa = os.path.join(ruta_carpeta, archivo)

        try:
            with open(ruta_completa, 'r', encoding='utf-8') as texto:
                texto_discurso = texto.read()
            
            # Creamos el hash SHA-256 del discurso el cual será utilizado como id_unico en MongoDB
            hash_id_unico = hashlib.sha256(texto_discurso.encode('utf-8')).hexdigest()

            # Creamos el embedding del texto
            vector_embedding = modelo_transformer.encode(texto_discurso).tolist()

            # Armamos el JSON que se guardará en la BD
            documento_final = {
                "_id": hash_id_unico,
                "nombre_archivo": archivo,
                "texto": texto_discurso,
                "embedding": vector_embedding
            }

            # Insertamos el nuevo JSON creado a la base de datos
            coleccion.insert_one(documento_final)
            print(f"  ✅ Procesado e insertado: {archivo}")

        except Exception as e:
            print(f"❌ Error al procesar el archivo {archivo}: {e}")

print("==============  Termino del procesamiento de los discursos  ==============")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

==============  Inicio del procesamiento de discursos  ==============
  ✅ Procesado e insertado: 100020.txt
  ✅ Procesado e insertado: 100033.txt
  ✅ Procesado e insertado: 100172.txt
  ✅ Procesado e insertado: 100178.txt
  ✅ Procesado e insertado: 100227.txt
  ✅ Procesado e insertado: 100260.txt
  ✅ Procesado e insertado: 100296.txt
  ✅ Procesado e insertado: 100302.txt
  ✅ Procesado e insertado: 100385.txt
  ✅ Procesado e insertado: 100422.txt
  ✅ Procesado e insertado: 100454.txt
  ✅ Procesado e insertado: 100509.txt
  ✅ Procesado e insertado: 100547.txt
  ✅ Procesado e insertado: 100603.txt
  ✅ Procesado e insertado: 100990.txt
  ✅ Procesado e insertado: 101030.txt
  ✅ Procesado e insertado: 101041.txt
  ✅ Procesado e insertado: 101075.txt
  ✅ Procesado e insertado: 101103.txt
  ✅ Procesado e insertado: 101169.txt
  ✅ Procesado e insertado: 101255.txt
  ✅ Procesado e insertado: 101299.txt
  ✅ Procesado e insertado: 101310.txt
  ✅ Procesado e insertado: 101337.txt
  ✅ Procesado e in

## 4. Consulta textual y similitud coseno.

In [122]:
# Definimos funcion se similitud coseno
def similitud_coseno(v1, v2):
    arr1 = np.array(v1)
    arr2 = np.array(v2)
    
    num = np.dot(arr1, arr2)
    den = np.linalg.norm(arr1) * np.linalg.norm(arr2)
    
    if den == 0:
        return 0.0
    
    return float(num / den)

def buscar_top_5(consulta_texto):
    # 1. Generar embedding de la consulta
    embedding_consulta = modelo_transformer.encode(consulta_texto).tolist()
    
    # 2. Traer todos los documentos de la base de datos
    documentos = list(coleccion.find({}, {"_id": 1, "nombre_archivo": 1, "texto": 1, "embedding": 1}))
    
    if not documentos:
        print("La base de datos está vacía. Ejecuta primero el script de preprocesamiento.")
        return
    
    # 3. Calcular la similitud para cada documento
    resultados = []
    for doc in documentos:
        similitud = similitud_coseno(embedding_consulta, doc["embedding"])

        resultados.append({
            "_id": doc["_id"],
            "nombre_archivo": doc.get("nombre_archivo", "Desconocido"),
            "texto_corto": doc["texto"][:200] + "...",
            "similitud": similitud
        })
    
    # 4. Ordenar de mayor a menor segun el puntaje de similitud
    resultados_ordenados = sorted(resultados, key=lambda x: x["similitud"], reverse=True)

    resultados_ordenados = resultados_ordenados[:5]
    
    # 5. Mostrar los Top 5 resultados por consola
    print(f"\n=== Top 5 documentos más similares a '{consulta_texto}' ===")

    cont = 1
    for resultado in resultados_ordenados:
        print()
        print(f"-- [Top {cont}] - Similitud: {resultado['similitud']:.4f}")
        print(f"    ID (SHA-256): {resultado['_id']}")
        print(f"    Archivo: {resultado['nombre_archivo']}")
        print(f"    Extracto: {resultado['texto_corto']}")
        print("-" * 60)

        cont += 1


Probamos los textos para consultar

In [123]:
textos_consulta = [
    "Chile necesita crecer economicamente para reducir la pobreza",
    "las pensiones de los adultos mayores son insuficientes",
    "la violencia en la Araucanía debe terminar con dialogo",
    "necesitamos una nueva constitucion que una a los chilenos"
]

for texto in textos_consulta:
    buscar_top_5(texto)


=== Top 5 documentos más similares a 'Chile necesita crecer economicamente para reducir la pobreza' ===

-- [Top 1] - Similitud: 0.7421
    ID (SHA-256): 01ddb8155a49edeae662dd55e56e027c92b3b61209977a7adb775b48562631f4
    Archivo: 88900.txt
    Extracto: Amigas y amigos, muy buenos días:

 

Quiero empezar por decir que, la misión que nuestra generación debe cumplir es transformar a Chile en un país desarrollado, sin pobreza, en que todos tengamos las...
------------------------------------------------------------

-- [Top 2] - Similitud: 0.7401
    ID (SHA-256): cc54693129b943fd69a182553701e40055517f811fc3512e5f0d19959e99bbd6
    Archivo: 84594.txt
    Extracto: Muy buenos días, amigas y amigos:

 

Estamos muy de acuerdo con las palabras del presidente de FEDETUR.

 

Quiero empezar por destacar que la gran misión de nuestro Gobierno es lograr que Chile, la ...
------------------------------------------------------------

-- [Top 3] - Similitud: 0.7393
    ID (SHA-256): 7fbb03d8eb8

## 5. Probamos consistencia

In [124]:
primary    = MongoClient("mongodb://localhost:30001/?directConnection=true")
secondary1 = MongoClient("mongodb://localhost:30002/?directConnection=true")
secondary2 = MongoClient("mongodb://localhost:30003/?directConnection=true")

for nombre, cliente in [("Primary :30001", primary), ("Secondary1 :30002", secondary1), ("Secondary2 :30003", secondary2)]:
    try:
        db = cliente.get_database("Política", read_preference=ReadPreference.SECONDARY_PREFERRED)
        count = coleccion.count_documents({})
        print(f"✅ {nombre}: {count} documentos")
    except Exception as e:
        print(f"❌ {nombre}: {e}")

✅ Primary :30001: 679 documentos
✅ Secondary1 :30002: 679 documentos
✅ Secondary2 :30003: 679 documentos


## 6. Probamos disponibilidad

### 6.1. Bajamos mongo-primary

> Bajamos el nodo primario llamado "mongo-primary"

In [125]:
subprocess.run(["docker", "stop", "mongo-primary"])
print("mongo-primary bajado")

mongo-primary bajado


> Consultamos disponibilidad

In [126]:
documentos = coleccion.find({}, {"_id": 0, "nombre_archivo": 1, "texto": 1}).limit(3)

for doc in documentos:
    print(f"Nombre archivo: {doc['nombre_archivo']}")
    print(f"Resumen texto:  {doc['texto'][:100]}...")
    print("-" * 60)

Nombre archivo: 100020.txt
Resumen texto:  Muy buenos días:

 

Disfruten de este Sol de invierno. Gabriela Mistral decía que “hay que amar a n...
------------------------------------------------------------
Nombre archivo: 100033.txt
Resumen texto:  Muy buenos días:

 

Blanca lo dijo, éste es un sueño cumplido y es un momento de reflexión. Uno se ...
------------------------------------------------------------
Nombre archivo: 100172.txt
Resumen texto:  Muy buenas tardes:

 

Señor Ministro, señor Intendente, señor Senador, señor Presidente del Directo...
------------------------------------------------------------


> Insertamos documento de prueba

In [127]:
try:
    coleccion.insert_one({
        "nombre_archivo": "prueba_2nodos.txt",
        "texto": "Hay 2 nodos"
    })

    print("Dato insertado")
except:
    print("Error al insertar en la colección de Discursos")



Dato insertado


### 6.2. Bajamos mongo-secondary1

> Bajamos otro nodo secundario llamado "mongo-secondary1"

In [128]:
subprocess.run(["docker", "stop", "mongo-secondary1"])
print("mongo-secondary1 bajado")

mongo-secondary1 bajado


> Consultamos

In [131]:
try:
    documentos = coleccion.find({}, {"_id": 0, "nombre_archivo": 1, "texto": 1}).limit(3)

    for doc in documentos:
        print(f"Nombre archivo: {doc['nombre_archivo']}")
        print(f"Resumen texto:  {doc['texto'][:100]}...")
        print("-" * 60)
except Exception as e:
    print(f"Cluster sin quorum, no hay primary disponible: {e}")

Cluster sin quorum, no hay primary disponible: No replica set members match selector "Primary()", Timeout: 30s, Topology Description: <TopologyDescription id: 6a1b6769fc6c35e348d70394, topology_type: ReplicaSetNoPrimary, servers: [<ServerDescription ('mongo-primary', 30001) server_type: Unknown, rtt: None, error=AutoReconnect('mongo-primary:30001: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>, <ServerDescription ('mongo-secondary1', 30002) server_type: Unknown, rtt: None, error=AutoReconnect('mongo-secondary1:30002: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>, <ServerDescription ('mongo-secondary2', 30003) server_type: RSSecondary, rtt: 0.00289690084997898>]>


### 6.3. Bajamos mongo-secondary2

> Bajamos el último nodo "mongo-secondary2"

In [132]:
subprocess.run(["docker", "stop", "mongo-secondary2"])
print("mongo-secondary2 bajado")

mongo-secondary2 bajado


> Hacemos consulta (Deberia dar error porque no hay nodos disponibles)

In [133]:
try:
    documentos = coleccion.find({}, {"_id": 0, "nombre_archivo": 1, "texto": 1}).limit(3)

    for doc in documentos:
        print(f"Nombre archivo: {doc['nombre_archivo']}")
        print(f"Resumen texto:  {doc['texto'][:100]}...")
        print("-" * 60)
except Exception as e:
    print(f"Cluster sin quorum, no hay primary disponible: {e}")

Cluster sin quorum, no hay primary disponible: mongo-secondary1:30002: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms),mongo-primary:30001: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms),mongo-secondary2:30003: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 6a1b6769fc6c35e348d70394, topology_type: ReplicaSetNoPrimary, servers: [<ServerDescription ('mongo-primary', 30001) server_type: Unknown, rtt: None, error=AutoReconnect('mongo-primary:30001: [WinError 10061] No se puede establecer una co

### 6.4. Levantamos los nodos

> Ahora levantamos todos los nodos otra vez y verificamos que los datos que insertamos cuando habían nodos caídos se repliquen

In [134]:
subprocess.run(["docker", "start", "mongo-primary"])
subprocess.run(["docker", "start", "mongo-secondary1"])
subprocess.run(["docker", "start", "mongo-secondary2"])

CompletedProcess(args=['docker', 'start', 'mongo-secondary2'], returncode=0)

> Esperar que se levanten correctamente los nodos

In [135]:
for nombre, cliente in [("Primary :30001", primary), ("Secondary1 :30002", secondary1), ("Secondary2 :30003", secondary2)]:
    try:
        doc_test = coleccion.find_one({"texto": "Hay 2 nodos"})

        if doc_test:
            print(f"   'Archivo prueba_2nodos.txt    encontrado en nodo {nombre}")
        else: 
            print(f"   'Archivo prueba_2nodos.txt NO encontrado en nodo {nombre}")

    except Exception as e:
        print(f"{nombre} CAÍDO")
        
    print("-" * 50)

   'Archivo prueba_2nodos.txt    encontrado en nodo Primary :30001
--------------------------------------------------
   'Archivo prueba_2nodos.txt    encontrado en nodo Secondary1 :30002
--------------------------------------------------
   'Archivo prueba_2nodos.txt    encontrado en nodo Secondary2 :30003
--------------------------------------------------
